In [23]:
from Bio import SeqIO
from collections import Counter
import random

# === FUNCTION: Mutation Percentage Calculator ===
def calculate_mutation_percentage_and_types(segment_alignment, reference_segment):
    total_positions = len(reference_segment)
    mutation_count = 0
    mutation_types = Counter()
    mutation_positions = [0] * total_positions

    valid_nucleotides = {'a', 't', 'g', 'c'}

    for seq in segment_alignment:
        for i in range(total_positions):
            ref_base = reference_segment[i]
            query_base = seq[i]

            # Count insertions or deletions
            if ref_base == '-' or query_base == '-':
                mutation_count += 1
                mutation_types[(query_base, ref_base)] += 1
                mutation_positions[i] += 1
            # Count substitutions only if both are valid A/T/G/C
            elif ref_base in valid_nucleotides and query_base in valid_nucleotides and ref_base != query_base:
                mutation_count += 1
                mutation_types[(query_base, ref_base)] += 1
                mutation_positions[i] += 1
            # Else: do not count

    mutation_percentage = (mutation_count / (total_positions * len(segment_alignment))) * 100
    return mutation_percentage, mutation_types, mutation_positions

In [25]:
# === LOAD MULTIPLE SEQUENCE ALIGNMENT FROM FASTA FILE ===
fasta_path = "orf1b-nsp14.fas"  # Replace with actual filename
records = list(SeqIO.parse(fasta_path, "fasta"))
genome_lines = [str(rec.seq).lower() for rec in records]

num_sequences = len(genome_lines)
print(f"Loaded {num_sequences} aligned sequences from FASTA.")

Loaded 6465 aligned sequences from FASTA.


In [27]:
# All targets coordinates in 1-based inclusive format
tr_coords = [
    (738, 757),
    (809, 832),
    (849, 869)
]

In [29]:
tr_coords = [(start - 1, end) for start, end in tr_coords]

In [31]:
# Build a set of all target-covered positions
tr_mask = set()
for start, end in tr_coords:
    tr_mask.update(range(start - 1, end))  # 0-indexed inclusive

segment_length = 26    #Change as per the length of the target sequences
valid_starts = []

# Determine all non-overlapping same length windows that do not touch target regions
for i in range(0, len(genome_lines[0]) - segment_length + 1, segment_length):
    segment_range = set(range(i, i + segment_length))
    if segment_range.isdisjoint(tr_mask):
        valid_starts.append(i)

print(f"Total valid non-overlapping non-Target windows: {len(valid_starts)}")

Total valid non-overlapping non-TR windows: 55


In [33]:
mutation_rates = []

for idx, start in enumerate(valid_starts):
    segment_alignment = [seq[start:start + segment_length] for seq in genome_lines]
    reference_segment = segment_alignment[0]

    mutation_percentage, _, _ = calculate_mutation_percentage_and_types(segment_alignment, reference_segment)
    mutation_rates.append(mutation_percentage)

In [34]:
import pandas as pd

end_positions = [start + segment_length - 1 for start in valid_starts]

results_df = pd.DataFrame({
    'Segment_Number': range(1, len(valid_starts)+1),
    'Start_Position': valid_starts,
    'End_Position': end_positions,
    'Mutation_Percentage': mutation_rates
})

results_df.to_excel("non_TR_mutation_rates_wash_S.xlsx", index=False)

In [37]:
# === Summary statistics ===
mean_rate = sum(mutation_rates) / len(mutation_rates)
min_rate = min(mutation_rates)
max_rate = max(mutation_rates)

print("\n--- Summary ---")
print(f"Mean mutation rate: {avg_rate:.4f}%")
print(f"Minimum mutation rate: {min_rate:.4f}%")
print(f"Maximum mutation rate: {max_rate:.4f}%")


--- Summary ---
Average mutation rate: 0.1219%
Minimum mutation rate: 0.0000%
Maximum mutation rate: 3.8295%
